In [1]:
import pandas as pd
from pathlib import Path
import os
from os.path import join as opj
from tqdm.auto import tqdm
import ast
import numpy as np
import re
from collections import Counter

In [2]:
BASE_DIR = os.getcwd()  
PARENT_DIR = os.path.dirname(BASE_DIR)  
DATASET_DIR = os.path.join(PARENT_DIR, "data")

In [3]:
i = 0
# read dataset
shein_data = pd.read_csv(opj(DATASET_DIR, "shein_sample_5000.csv"), sep=";", encoding="ISO-8859-1")
for i in tqdm(list(range(len(shein_data)))):
    desc, images = shein_data.iloc[i][["description", "images"]]
    
    # read description and images info as lists
    desc, images = ast.literal_eval(desc), ast.literal_eval(images)
    shein_data.loc[i, ["description", "images"]] = [desc, images]

  0%|          | 0/5000 [00:00<?, ?it/s]

In [4]:
shein_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   url          5000 non-null   object
 1   name         5000 non-null   object
 2   sku          5000 non-null   object
 3   price        5000 non-null   object
 4   size         4998 non-null   object
 5   brand        2428 non-null   object
 6   description  5000 non-null   object
 7   images       5000 non-null   object
dtypes: object(8)
memory usage: 312.6+ KB


In [5]:
shein_data.head(20)

,url,name,sku,price,size,brand,description,images
0,https://us.shein.com/Flower-Decor-Face-Coverin...,Flower Decor Face Covering Chain,SKU: sc2302055742295937,$2.50,one-size,QingSang Accessories Apparel Accessories,"[{'Color': 'Multicolor'}, {'Material': 'PMMA'}]",[https://storage.yandexcloud.net/clothes-and-w...
1,https://us.shein.com/Lace-Up-Bodycon-Skirt-p-7...,Lace Up Bodycon Skirt,SKU: sw2111151588254852,$7.25,"XS(2),S(4),M(6),L(8/10)",NaN,"[{'Color': 'Black'}, {'Style': 'Sexy'}, {'Patt...",[https://storage.yandexcloud.net/clothes-and-w...
2,https://us.shein.com/High-Waist-Twist-Front-Sh...,High Waist Twist Front Shorts,SKU: sw2211215551435599,$8.99,"S(4),M(6),L(8/10),XL(12)",NaN,"[{'Color': 'Royal Blue'}, {'Style': 'Casual'},...",[https://storage.yandexcloud.net/clothes-and-w...
3,https://us.shein.com/ROMWE-Sweetness-Cherry-La...,ROMWE Sweetness Cherry Lace Cami Top,SKU: rw2203161166248409,$7.00,"XS(2),S(4),M(6),L(8/10),XL(12),XXL(14)",ROMWE,"[{'Color': 'Red and White'}, {'Style': 'Casual...",[https://storage.yandexcloud.net/clothes-and-w...
4,https://us.shein.com/Cut-Out-Criss-Cross-One-P...,Cut Out Criss Cross One Piece Swimsuit,SKU: sw2108269195581516,$9.50,"S(4),M(6),L(8/12)",NaN,"[{'Color': 'Lilac Purple'}, {'Style': 'Elegant...",[https://storage.yandexcloud.net/clothes-and-w...
5,https://us.shein.com/Tie-Backless-Halter-Trian...,Tie Backless Halter Triangle V-string Bikini S...,SKU: sw2211254128128287,$10.49,"S(4),M(6),L(8/12)",NaN,"[{'Color': 'White'}, {'Pattern Type': 'Plain'}...",[https://storage.yandexcloud.net/clothes-and-w...
6,https://us.shein.com/Solid-Tie-Side-Cover-Up-S...,Solid Tie Side Cover Up Skirt,SKU: sw2202158913200151,$4.49,one-size,NaN,"[{'Color': 'Black'}, {'Pattern Type': 'Plain'}...",[https://storage.yandexcloud.net/clothes-and-w...
7,https://us.shein.com/ROMWE-PUNK-Tie-Dye-Letter...,ROMWE PUNK Tie Dye & Letter Graphic Notched Ne...,SKU: sw2205252404438114,$6.99,"XS(2),S(4),M(6),L(8/10)",ROMWE,"[{'Color': 'Brown'}, {'Style': 'Casual'}, {'Pa...",[https://storage.yandexcloud.net/clothes-and-w...
8,https://us.shein.com/O-ring-Detail-Halter-Knot...,O-ring Detail Halter Knot Side Triangle Thong ...,SKU: sw2202159072898493,$11.49,"S(4),M(6),L(8/12),XL(14)",NaN,"[{'Color': 'Mocha Brown'}, {'Style': 'Sexy'}, ...",[https://storage.yandexcloud.net/clothes-and-w...
9,https://us.shein.com/Bow-Bead-Decor-Face-Cover...,Bow & Bead Decor Face Covering Chain,SKU: sc2302052787833869,$2.80,one-size,QingSang Accessories Apparel Accessories,"[{'Color': 'Multicolor'}, {'Material': 'PMMA'}]",[https://storage.yandexcloud.net/clothes-and-w...


In [6]:
#dropping unwanted columns
shein_data = shein_data.drop(columns=['url'])

In [7]:
shein_data.head()

,name,sku,price,size,brand,description,images
0,Flower Decor Face Covering Chain,SKU: sc2302055742295937,$2.50,one-size,QingSang Accessories Apparel Accessories,"[{'Color': 'Multicolor'}, {'Material': 'PMMA'}]",[https://storage.yandexcloud.net/clothes-and-w...
1,Lace Up Bodycon Skirt,SKU: sw2111151588254852,$7.25,"XS(2),S(4),M(6),L(8/10)",NaN,"[{'Color': 'Black'}, {'Style': 'Sexy'}, {'Patt...",[https://storage.yandexcloud.net/clothes-and-w...
2,High Waist Twist Front Shorts,SKU: sw2211215551435599,$8.99,"S(4),M(6),L(8/10),XL(12)",NaN,"[{'Color': 'Royal Blue'}, {'Style': 'Casual'},...",[https://storage.yandexcloud.net/clothes-and-w...
3,ROMWE Sweetness Cherry Lace Cami Top,SKU: rw2203161166248409,$7.00,"XS(2),S(4),M(6),L(8/10),XL(12),XXL(14)",ROMWE,"[{'Color': 'Red and White'}, {'Style': 'Casual...",[https://storage.yandexcloud.net/clothes-and-w...
4,Cut Out Criss Cross One Piece Swimsuit,SKU: sw2108269195581516,$9.50,"S(4),M(6),L(8/12)",NaN,"[{'Color': 'Lilac Purple'}, {'Style': 'Elegant...",[https://storage.yandexcloud.net/clothes-and-w...


In [8]:
#finding all of the brand names
shein_data['brand'].unique().tolist()

['QingSang\xa0Accessories Apparel Accessories',
 nan,
 'ROMWE',
 'SHEIN ICON',
 'Bikinx',
 'DAZY',
 'SHEIN SXY',
 'DiFanYa Apparel Accessories',
 'YouyueSportsGoodsFactory Apparel Accessories',
 'SHEIN EZwear',
 'SHEIN X VICTOR VON SCHWARZ',
 'SHEIN X Kovid Kapoor',
 'SHEIN X VOON',
 'SHEIN X EMILY REED',
 'SHEIN Frenchy',
 'EMERY ROSE',
 'SHEIN BAE',
 'DANDING Apparel Accessories',
 'SHEIN BIZwear',
 'SHEIN X Cecile Betbeze',
 'SHEIN VCAY',
 'Bazun',
 'MOTF',
 'SHEIN Unity',
 'Zhanyan Apparel Accessories',
 'SHEIN X VALHALLA',
 'SHEIN X No?-Liam',
 'SHEIN X Lindsey Kate',
 'SHEIN Modely',
 'SHEIN X (in)SALT-Y(d)',
 'YHOLF Apparel Accessories',
 'InnNew Apparel Accessories',
 'Xinshi Apparel Accessories',
 'Yuxiang Apparel Accessories',
 'SHEIN X Firefall',
 'SHEIN X Sayndo',
 'Meanbeauty',
 'SHEIN X VERO DIAZ',
 'Baozhining Apparel Accessories',
 'SHEIN BASICS',
 'SHEINNeu',
 'SHEIN Tall',
 'Jhdxjwmy',
 'Double Crazy',
 'SHEIN X GottaBeJelly',
 'evoluSHEIN',
 'SHEIN X BAW Collection',

In [9]:
#suspecting NaN values in the brand column are from the regular line
brands_to_find = ['shein', 'SheIn', 'SHEIN','She In', 'she in', 'SHE IN']
found = shein_data['brand'].isin(brands_to_find).any()
print(found)

False


In [10]:
#replacing NaN values with SHEIN in the brand column and checking
shein_data['brand'] = shein_data['brand'].replace(np.nan, 'SHEIN')
brands_to_find = ['SHEIN']
found = shein_data['brand'].isin(brands_to_find).any()
print(found)

True


In [11]:
#changing price column to a float 
shein_data['price'] = shein_data['price'].str.replace('$', '', regex=False)
shein_data['price'] = pd.to_numeric(shein_data['price'], errors='coerce')
shein_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   name         5000 non-null   object 
 1   sku          5000 non-null   object 
 2   price        4999 non-null   float64
 3   size         4998 non-null   object 
 4   brand        5000 non-null   object 
 5   description  5000 non-null   object 
 6   images       5000 non-null   object 
dtypes: float64(1), object(6)
memory usage: 273.6+ KB


In [12]:
#investigating name column
print(shein_data['name'].head(100).to_list())

['Flower Decor Face Covering Chain', 'Lace Up Bodycon Skirt', 'High Waist Twist Front Shorts', 'ROMWE Sweetness Cherry Lace Cami Top', 'Cut Out Criss Cross One Piece Swimsuit', 'Tie Backless Halter Triangle V-string Bikini Swimsuit', 'Solid Tie Side Cover Up Skirt', 'ROMWE PUNK Tie Dye & Letter Graphic Notched Neck Bell Sleeve Crop Tee', 'O-ring Detail Halter Knot Side Triangle Thong Bikini Swimsuit', 'Bow & Bead Decor Face Covering Chain', 'SHEIN ICON Buckle Tape Cut Out Lace Up Bootcut Pants', 'ROMWE PUNK Grunge High Waist Floral & Baroque Print Lettuce Trim Skirt', 'Paperbag Waist Wide Leg Pants', 'Contrast Mesh Cut Out Halter One Piece Swimsuit', 'Color Block Triangle Thong Bikini Swimsuit', 'Split Thigh Cover Up', 'ROMWE Eyelet Embroidery Cami', 'Bikinx Rib Triangle High Cut Bikini Swimsuit', 'ROMWE Kawaii Bow Decor Face Covering', 'SHEIN Slant Pocket Belted PU Leather Pants', 'Geo Print Wide Waistband Biker Shorts', 'ROMWE Fairycore Drawstring Waist Tartan Skirt', 'Glitter Lace U

In [13]:
#category function
def assign_category(df, col):

    conditions = [
        shein_data['name'].str.contains('Skirt', case=False, na=False),
        shein_data['name'].str.contains('Dress', case=False, na=False),
        shein_data['name'].str.contains('Shorts', case=False, na=False),
        shein_data['name'].str.contains('skort', case=False, na=False),
        shein_data['name'].str.contains('Top|Cami|Tee', case=False, na=False),
        shein_data['name'].str.contains('Pants|Trousers|Joggers', case=False, na=False),
        shein_data['name'].str.contains('Swimsuit', case=False, na=False),
        shein_data['name'].str.contains('Chain', case=False, na=False),
    ]

    categories = ['Skirt', 'Dress', 'Shorts', 'Skort', 'Top', 'Pants', 'Swimsuit', 'Jewelry']

    shein_data['category'] = np.select(conditions, categories, default='Other')
    return df

In [14]:
#applying the function to create a category
shein_data = assign_category(shein_data, 'name')
shein_data.head()

,name,sku,price,size,brand,description,images,category
0,Flower Decor Face Covering Chain,SKU: sc2302055742295937,2.50,one-size,QingSang Accessories Apparel Accessories,"[{'Color': 'Multicolor'}, {'Material': 'PMMA'}]",[https://storage.yandexcloud.net/clothes-and-w...,Jewelry
1,Lace Up Bodycon Skirt,SKU: sw2111151588254852,7.25,"XS(2),S(4),M(6),L(8/10)",SHEIN,"[{'Color': 'Black'}, {'Style': 'Sexy'}, {'Patt...",[https://storage.yandexcloud.net/clothes-and-w...,Skirt
2,High Waist Twist Front Shorts,SKU: sw2211215551435599,8.99,"S(4),M(6),L(8/10),XL(12)",SHEIN,"[{'Color': 'Royal Blue'}, {'Style': 'Casual'},...",[https://storage.yandexcloud.net/clothes-and-w...,Shorts
3,ROMWE Sweetness Cherry Lace Cami Top,SKU: rw2203161166248409,7.00,"XS(2),S(4),M(6),L(8/10),XL(12),XXL(14)",ROMWE,"[{'Color': 'Red and White'}, {'Style': 'Casual...",[https://storage.yandexcloud.net/clothes-and-w...,Top
4,Cut Out Criss Cross One Piece Swimsuit,SKU: sw2108269195581516,9.50,"S(4),M(6),L(8/12)",SHEIN,"[{'Color': 'Lilac Purple'}, {'Style': 'Elegant...",[https://storage.yandexcloud.net/clothes-and-w...,Swimsuit


In [15]:
print(shein_data['images'].head(3).to_list())

[['https://storage.yandexcloud.net/clothes-and-wildberries/clothes-parsing-dataset/shein/2023/02/06/16756668443ff7011f5eb2c82814c0884603d2947a_thumbnail_600x.webp', 'https://storage.yandexcloud.net/clothes-and-wildberries/clothes-parsing-dataset/shein/2023/02/06/1675666836e4bc53f352ed55ad72d102f3c918ec47_thumbnail_600x.webp', 'https://storage.yandexcloud.net/clothes-and-wildberries/clothes-parsing-dataset/shein/2023/02/06/16756668385c70273ea378a50508a68de521c6d226_thumbnail_600x.webp', 'https://storage.yandexcloud.net/clothes-and-wildberries/clothes-parsing-dataset/shein/2023/02/06/1675666841a31c344ad7169f3fe6ed57c017b171de_thumbnail_600x.webp', 'https://storage.yandexcloud.net/clothes-and-wildberries/clothes-parsing-dataset/shein/2023/02/06/1675666843c8b780757e1ea957d597e89eced1f6a5_thumbnail_600x.webp', 'https://storage.yandexcloud.net/clothes-and-wildberries/clothes-parsing-dataset/shein/2023/02/06/16756668443ff7011f5eb2c82814c0884603d2947a_thumbnail_600x.webp', 'https://storage.yan

In [16]:
shein_data['image_count'] = shein_data['images'].apply(len)
shein_data.drop(columns=['images'], inplace = True)#dropping images column
shein_data.head()

,name,sku,price,size,brand,description,category,image_count
0,Flower Decor Face Covering Chain,SKU: sc2302055742295937,2.50,one-size,QingSang Accessories Apparel Accessories,"[{'Color': 'Multicolor'}, {'Material': 'PMMA'}]",Jewelry,7
1,Lace Up Bodycon Skirt,SKU: sw2111151588254852,7.25,"XS(2),S(4),M(6),L(8/10)",SHEIN,"[{'Color': 'Black'}, {'Style': 'Sexy'}, {'Patt...",Skirt,9
2,High Waist Twist Front Shorts,SKU: sw2211215551435599,8.99,"S(4),M(6),L(8/10),XL(12)",SHEIN,"[{'Color': 'Royal Blue'}, {'Style': 'Casual'},...",Shorts,9
3,ROMWE Sweetness Cherry Lace Cami Top,SKU: rw2203161166248409,7.00,"XS(2),S(4),M(6),L(8/10),XL(12),XXL(14)",ROMWE,"[{'Color': 'Red and White'}, {'Style': 'Casual...",Top,6
4,Cut Out Criss Cross One Piece Swimsuit,SKU: sw2108269195581516,9.50,"S(4),M(6),L(8/12)",SHEIN,"[{'Color': 'Lilac Purple'}, {'Style': 'Elegant...",Swimsuit,10


In [17]:
#exploring - checking if the same product is listed more than once using sku
shein_data['sku'].nunique()

5000

In [18]:
#exploring - checking if the same product is listed more than once using name
shein_data['name'].nunique()

4423

In [19]:
#converting size into a string
shein_data['size'] = shein_data['size'].apply(str)

In [20]:
#creating an additional size column that cleanly displays sizes
def parse_sizes(size_str):
    sizes = size_str.split(',')
    return ','.join([s.split('(')[0] for s in sizes])

shein_data['cleaned_size'] = shein_data['size'].apply(parse_sizes)
shein_data.drop(columns=['size'], inplace=True)
shein_data.rename(columns={'cleaned_size': 'sizes'}, inplace=True)

shein_data.head()

,name,sku,price,brand,description,category,image_count,sizes
0,Flower Decor Face Covering Chain,SKU: sc2302055742295937,2.50,QingSang Accessories Apparel Accessories,"[{'Color': 'Multicolor'}, {'Material': 'PMMA'}]",Jewelry,7,one-size
1,Lace Up Bodycon Skirt,SKU: sw2111151588254852,7.25,SHEIN,"[{'Color': 'Black'}, {'Style': 'Sexy'}, {'Patt...",Skirt,9,"XS,S,M,L"
2,High Waist Twist Front Shorts,SKU: sw2211215551435599,8.99,SHEIN,"[{'Color': 'Royal Blue'}, {'Style': 'Casual'},...",Shorts,9,"S,M,L,XL"
3,ROMWE Sweetness Cherry Lace Cami Top,SKU: rw2203161166248409,7.00,ROMWE,"[{'Color': 'Red and White'}, {'Style': 'Casual...",Top,6,"XS,S,M,L,XL,XXL"
4,Cut Out Criss Cross One Piece Swimsuit,SKU: sw2108269195581516,9.50,SHEIN,"[{'Color': 'Lilac Purple'}, {'Style': 'Elegant...",Swimsuit,10,"S,M,L"


In [21]:
print(shein_data['description'].head(3).to_list())

[[{'Color': 'Multicolor'}, {'Material': 'PMMA'}], [{'Color': 'Black'}, {'Style': 'Sexy'}, {'Pattern Type': 'Plain'}, {'Details': 'Lace Up, Grommet Eyelet'}, {'Waist Line': 'High Waist'}, {'Length': 'Mini'}, {'Type': 'Bodycon'}, {'Fabric': 'High Stretch'}, {'Material': 'Fabric'}, {'Composition': '95% Polyester, 5% Elastane'}, {'Care Instructions': 'Machine wash or professional dry clean'}, {'Sheer': 'No'}], [{'Color': 'Royal Blue'}, {'Style': 'Casual'}, {'Pattern Type': 'Plain'}, {'Type': 'Straight Leg'}, {'Closure Type': 'Zipper Fly'}, {'Details': 'Twist'}, {'Waist Line': 'Natural'}, {'Length': 'Short'}, {'Fit Type': 'Regular Fit'}, {'Fabric': 'Non-Stretch'}, {'Material': 'Woven Fabric'}, {'Composition': '100% Polyester'}, {'Care Instructions': 'Machine wash, do not dry clean'}, {'Sheer': 'No'}]]


In [22]:
# extracting color
def extract_color(desc):
    if isinstance(desc, list):
        for d in desc:
            if 'Color' in d:
                return d['Color']
    return None

shein_data['color'] = shein_data['description'].apply(extract_color)
shein_data.head()

,name,sku,price,brand,description,category,image_count,sizes,color
0,Flower Decor Face Covering Chain,SKU: sc2302055742295937,2.50,QingSang Accessories Apparel Accessories,"[{'Color': 'Multicolor'}, {'Material': 'PMMA'}]",Jewelry,7,one-size,Multicolor
1,Lace Up Bodycon Skirt,SKU: sw2111151588254852,7.25,SHEIN,"[{'Color': 'Black'}, {'Style': 'Sexy'}, {'Patt...",Skirt,9,"XS,S,M,L",Black
2,High Waist Twist Front Shorts,SKU: sw2211215551435599,8.99,SHEIN,"[{'Color': 'Royal Blue'}, {'Style': 'Casual'},...",Shorts,9,"S,M,L,XL",Royal Blue
3,ROMWE Sweetness Cherry Lace Cami Top,SKU: rw2203161166248409,7.00,ROMWE,"[{'Color': 'Red and White'}, {'Style': 'Casual...",Top,6,"XS,S,M,L,XL,XXL",Red and White
4,Cut Out Criss Cross One Piece Swimsuit,SKU: sw2108269195581516,9.50,SHEIN,"[{'Color': 'Lilac Purple'}, {'Style': 'Elegant...",Swimsuit,10,"S,M,L",Lilac Purple


In [23]:
#display all the colors
shein_data['color'].unique()

array(['Multicolor', 'Black', 'Royal Blue', 'Red and White',
       'Lilac Purple', 'White', 'Brown', 'Mocha Brown', 'Army Green',
       'Navy Blue', 'Teal Blue', 'Pink', 'Rust Brown', 'Chocolate Brown',
       'Silver', 'Green', 'Khaki', 'Black and White', 'Lime Green',
       'Baby Pink', 'Violet Purple', 'Orange', 'Camel', 'Mint Green',
       'Coffee Brown', 'Apricot', 'Light Grey', 'Dusty Pink', 'Blue',
       'Hot Pink', 'Beige', 'Cadet Blue', 'Olive Green', 'Yellow',
       'Maroon', 'Blue and White', 'Grey', 'Dark Grey', 'Mint Blue',
       'Clear', 'Dark Green', 'Baby Blue', 'Red', 'Rusty Rose',
       'Burgundy', 'Burnt Orange', 'Dusty Blue', 'Gold', 'Coral Orange',
       'Mustard Yellow', 'Champagne', 'Mauve Purple', 'Purple', 'Redwood',
       'Red Violet', 'Medium Wash', 'Watermelon Pink', 'Bronze',
       'Rose Red', 'Ginger', 'Coral Pink', 'Light Wash', 'Dark Wash',
       'Dusty Purple'], dtype=object)

In [24]:
# check missing values
print(shein_data['color'].isnull().sum()) 

0


In [25]:
#adding color category from sub colors  
def group_colors(color):
    color_groups = {
        'Black': ['Black'],
        'Black and White': ['Black and White'],
        'Red': ['Red', 'Red Violet', 'Burgundy', 'Rose Red', 'Rusty Rose', 'Red and White'],
        'Pink': ['Watermelon Pink', 'Pink', 'Dusty Pink', 'Baby Pink', 'Coral Pink'],
        'Blue': ['Royal Blue', 'Navy Blue', 'Teal Blue', 'Blue', 'Cadet Blue', 'Blue and White', 'Light Wash', 'Dark Wash', 'Medium Wash', 'Dusty Blue'],
        'Yellow': ['Yellow', 'Mustard Yellow'],
        'White': ['White', 'Clear'],
        'Purple': ['Lilac Purple', 'Violet Purple', 'Mauve Purple', 'Purple', 'Dusty Purple'],
        'Green': ['Army Green', 'Green', 'Lime Green', 'Mint Green', 'Olive Green', 'Dark Green'],
        'Brown': ['Brown', 'Mocha Brown', 'Chocolate Brown', 'Rust Brown', 'Coffee Brown', 'Ginger', 'Camel', 'Apricot'],
        'Neautral': ['Beige', 'Khaki'],
        'Grey/Silver': ['Light Grey', 'Silver'],
        'Gold/Champagne': ['Gold', 'Champagne'],
        'Orange': ['Orange', 'Burnt Orange', 'Coral Orange'],
    }

    for group, colors in color_groups.items():
        if color in colors:
            return group
    return 'Multi'

shein_data['color_grouped'] = shein_data['color'].apply(group_colors)
shein_data.rename(columns={'color':'subcolor'}, inplace = True)
shein_data.head()

,name,sku,price,brand,description,category,image_count,sizes,subcolor,color_grouped
0,Flower Decor Face Covering Chain,SKU: sc2302055742295937,2.50,QingSang Accessories Apparel Accessories,"[{'Color': 'Multicolor'}, {'Material': 'PMMA'}]",Jewelry,7,one-size,Multicolor,Multi
1,Lace Up Bodycon Skirt,SKU: sw2111151588254852,7.25,SHEIN,"[{'Color': 'Black'}, {'Style': 'Sexy'}, {'Patt...",Skirt,9,"XS,S,M,L",Black,Black
2,High Waist Twist Front Shorts,SKU: sw2211215551435599,8.99,SHEIN,"[{'Color': 'Royal Blue'}, {'Style': 'Casual'},...",Shorts,9,"S,M,L,XL",Royal Blue,Blue
3,ROMWE Sweetness Cherry Lace Cami Top,SKU: rw2203161166248409,7.00,ROMWE,"[{'Color': 'Red and White'}, {'Style': 'Casual...",Top,6,"XS,S,M,L,XL,XXL",Red and White,Red
4,Cut Out Criss Cross One Piece Swimsuit,SKU: sw2108269195581516,9.50,SHEIN,"[{'Color': 'Lilac Purple'}, {'Style': 'Elegant...",Swimsuit,10,"S,M,L",Lilac Purple,Purple


In [26]:
def extract_attribute(desc_list, key):
    if isinstance(desc_list, list):
        for d in desc_list:
            if key in d:
                return d[key]
    return None

shein_data['material'] = shein_data['description'].apply(lambda x: extract_attribute(x, 'Material'))
shein_data['composition'] = shein_data['description'].apply(lambda x: extract_attribute(x, 'Composition'))

In [27]:
shein_data.head(10)

,name,sku,price,brand,description,category,image_count,sizes,subcolor,color_grouped,material,composition
0,Flower Decor Face Covering Chain,SKU: sc2302055742295937,2.50,QingSang Accessories Apparel Accessories,"[{'Color': 'Multicolor'}, {'Material': 'PMMA'}]",Jewelry,7,one-size,Multicolor,Multi,PMMA,None
1,Lace Up Bodycon Skirt,SKU: sw2111151588254852,7.25,SHEIN,"[{'Color': 'Black'}, {'Style': 'Sexy'}, {'Patt...",Skirt,9,"XS,S,M,L",Black,Black,Fabric,"95% Polyester, 5% Elastane"
2,High Waist Twist Front Shorts,SKU: sw2211215551435599,8.99,SHEIN,"[{'Color': 'Royal Blue'}, {'Style': 'Casual'},...",Shorts,9,"S,M,L,XL",Royal Blue,Blue,Woven Fabric,100% Polyester
3,ROMWE Sweetness Cherry Lace Cami Top,SKU: rw2203161166248409,7.00,ROMWE,"[{'Color': 'Red and White'}, {'Style': 'Casual...",Top,6,"XS,S,M,L,XL,XXL",Red and White,Red,Mesh Fabric,"70% Polyamide, 30% Spandex"
4,Cut Out Criss Cross One Piece Swimsuit,SKU: sw2108269195581516,9.50,SHEIN,"[{'Color': 'Lilac Purple'}, {'Style': 'Elegant...",Swimsuit,10,"S,M,L",Lilac Purple,Purple,Fabric,"80% Polyamide, 20% Elastane"
5,Tie Backless Halter Triangle V-string Bikini S...,SKU: sw2211254128128287,10.49,SHEIN,"[{'Color': 'White'}, {'Pattern Type': 'Plain'}...",Swimsuit,7,"S,M,L",White,White,None,None
6,Solid Tie Side Cover Up Skirt,SKU: sw2202158913200151,4.49,SHEIN,"[{'Color': 'Black'}, {'Pattern Type': 'Plain'}...",Skirt,6,one-size,Black,Black,Fabric,"95% Polyester, 5% Elastane"
7,ROMWE PUNK Tie Dye & Letter Graphic Notched Ne...,SKU: sw2205252404438114,6.99,ROMWE,"[{'Color': 'Brown'}, {'Style': 'Casual'}, {'Pa...",Top,9,"XS,S,M,L",Brown,Brown,Fabric,100% Polyester
8,O-ring Detail Halter Knot Side Triangle Thong ...,SKU: sw2202159072898493,11.49,SHEIN,"[{'Color': 'Mocha Brown'}, {'Style': 'Sexy'}, ...",Swimsuit,10,"S,M,L,XL",Mocha Brown,Brown,None,None
9,Bow & Bead Decor Face Covering Chain,SKU: sc2302052787833869,2.80,QingSang Accessories Apparel Accessories,"[{'Color': 'Multicolor'}, {'Material': 'PMMA'}]",Jewelry,7,one-size,Multicolor,Multi,PMMA,None


In [28]:
shein_data['material'].unique()

array(['PMMA', 'Fabric', 'Woven Fabric', 'Mesh Fabric', None,
       'Knitted Fabric', 'Fiber', 'PU Leather', 'Worsted', 'Lace',
       'Polyester', 'Sequins', 'Rib knit', 'Flannelette', 'Plastic',
       'Copper', 'Corduroy', 'Denim', 'Satin', 'Chiffon', 'Glass',
       'Tweed', 'Non-woven Fabric', 'Iron', 'Acrylic', 'Metallic',
       'Suedette', 'Synthetic Fiber', 'Zinc Alloy', 'Polymer Clay',
       'Copper Alloy', 'Linen', 'Polyresin', 'ABS', 'Stainless Steel',
       'Glitter', 'Wool-Like Fabric', 'Artificial Crystal', 'Iron Alloy',
       'Elastane', 'Polyamide', 'Mohair', 'Silk', 'Natural Crystal',
       'Flocking', 'Shell', 'Silicone', 'Embroidery', 'Paper'],
      dtype=object)

In [29]:
shein_data['composition'].unique()

array([None, '95% Polyester, 5% Elastane', '100% Polyester',
       '70% Polyamide, 30% Spandex', '80% Polyamide, 20% Elastane',
       '91% Polyester, 9% Elastane', '60% Viscose, 40% Polyester',
       '82% Polyester, 18% Elastane', '100% Acrylic',
       '94% Viscose, 6% Elastane', '94% Polyester, 6% Elastane',
       '95% Polyester, 5% Spandex', '82% Polyester, 18% Spandex',
       '100% Viscose', '97% Polyester, 3% Elastane',
       '94% Polyester, 6% Spandex', '92% Polyester, 8% Spandex',
       '93% Polyester, 7% Elastane', '100% Polyamide',
       '80% Polyester, 20% Viscose',
       '50% Rayon, 45% Polyester, 5% Elastane',
       '65% Viscose, 35% Polyester',
       '77% Polyester, 16% Viscose, 7% Elastane',
       '92% Polyamide, 8% Elastane', '100% Cotton',
       '80% Nylon, 20% Elastane',
       '62% Polyester, 33% Cotton, 5% Elastane', '65% Nylon, 35% Rayon',
       '96% Polyester, 4% Elastane', '97% Polyester, 3% Spandex',
       '50% Viscose, 28% Nylon, 22% Polyester', '

In [32]:
#number of words in 'name' column distribution 
shein_data['name_word_count'] = shein_data['name'].apply(lambda x: len(str(x).split()))
shein_data['name_word_count'].describe()

count    5000.000000
mean        6.949600
std         1.947567
min         3.000000
25%         5.000000
50%         7.000000
75%         8.000000
max        18.000000
Name: name_word_count, dtype: float64

In [31]:
#most common key words
all_words = shein_data['name'].dropna().apply(lambda x: re.findall(r'\b\w+\b', x.lower()))
flat_words = [word for sublist in all_words for word in sublist]
word_freq = Counter(flat_words)

most_common_words = word_freq.most_common(20)
print(most_common_words)

[('shein', 1653), ('skirt', 1223), ('print', 1095), ('pants', 807), ('waist', 771), ('bikini', 766), ('swimsuit', 731), ('hem', 636), ('face', 602), ('covering', 600), ('floral', 534), ('triangle', 523), ('x', 522), ('high', 517), ('leg', 508), ('chain', 501), ('solid', 468), ('halter', 454), ('shorts', 393), ('tie', 393)]
